# Save result pic as videos

In [11]:
import os
import cv2
from tqdm import tqdm

# folders
camera_pic_folders = ['randla', 'step']
output_folder = 'videos'  # where the .mp4 files will be written
os.makedirs(output_folder, exist_ok=True)

# parameters
fps = 16  # frames per second for the output video
# try 'mp4v' first; if your player can't open the result, try 'avc1' (H.264) or switch to .avi with 'XVID'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

def numeric_key(fn):
    """Sort by the numeric filename (without extension); fall back to lexicographic."""
    name, _ = os.path.splitext(fn)
    try:
        return int(name)
    except ValueError:
        return name  # fallback for non-numeric

for current_folder in tqdm(camera_pic_folders, desc="Building videos", unit="folder"):
    if not os.path.isdir(current_folder):
        print(f"Skip: '{current_folder}' is not a directory.")
        continue

    # each subfolder in current_folder becomes one video
    list_subfolders = [d for d in os.listdir(current_folder)
                       if os.path.isdir(os.path.join(current_folder, d))]
    if not list_subfolders:
        print(f"No subfolders found in '{current_folder}'.")
        continue

    for sub in tqdm(list_subfolders, desc=f"Building {current_folder}", unit="subfolder"):
        cam_dir = os.path.join(current_folder, sub)
        files = sorted(
            [f for f in os.listdir(cam_dir) if f.lower().endswith('.png')],
            key=numeric_key
        )

        if not files:
            print(f"No .png files in '{cam_dir}'. Skipping.")
            continue

        # determine frame size (W, H) correctly
        first_path = os.path.join(cam_dir, files[0])
        first_img = cv2.imread(first_path, cv2.IMREAD_COLOR)
        if first_img is None:
            print(f"Cannot read first image: {first_path}. Skipping.")
            continue
        H, W = first_img.shape[:2]  # shape returns (H, W, C)
        frame_size = (W, H)         # VideoWriter expects (width, height)

        # create output path mirroring current_folder structure
        out_dir = os.path.join(output_folder, current_folder)
        os.makedirs(out_dir, exist_ok=True)
        out_path = os.path.join(out_dir, f'{sub}.mp4')

        writer = cv2.VideoWriter(out_path, fourcc, fps, frame_size)
        if not writer.isOpened():
            print(f"Failed to open writer for: {out_path}")
            continue

        # write frames
        wrote = 0
        for fn in (files):
            img_path = os.path.join(cam_dir, fn)
            frame = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if frame is None:
                print(f"Warning: couldn't read {img_path}; skipping.")
                continue
            if frame.shape[1] != W or frame.shape[0] != H:
                # resize mismatched frames to the target size
                frame = cv2.resize(frame, frame_size, interpolation=cv2.INTER_AREA)
            writer.write(frame)
            wrote += 1

        writer.release()

        # if wrote == 0:
        #     print(f"No frames written for {out_path}. The video will be empty.")
        #     # optional: os.remove(out_path)
        # else:
        #     print(f"✅ Wrote {wrote} frames to {out_path}")


Building videos: 100%|██████████| 2/2 [00:49<00:00, 24.87s/folder]


# Combine Videos

In [4]:
import os
import re
import cv2
import numpy as np
from tqdm import tqdm
from collections import defaultdict

# -------- CONFIG --------
VIDEOS_ROOT    = "videos"
GROUPS         = ["step", "randla"]     # process both groups
OUT_SUBFMT     = "combine_{group}"      # per-group output subfolder
EXPECTED_COUNT = 5                       # we expect _1.._5

# Choose layout: "row" => 1x5, "2x3" => 2x3 grid (one empty slot)
LAYOUT_MODE    = "2x3"                  # "row" or "2x3"

# Visual tuning
TARGET_CELL_W  = 450
TARGET_CELL_H  = 288
GUTTER         = 6
BG_COLOR       = (0, 0, 0)              # BGR

# Labels
DRAW_LABELS    = True                   # turn labels on/off
LABEL_PADDING  = 8                      # inner padding for the label box
LABEL_MARGIN   = 10                     # distance from top-left corner
LABEL_BG_ALPHA = 0.6                    # 0..1 transparency of the label box

# Codec
FOURCC = cv2.VideoWriter_fourcc(*"mp4v")
# ------------------------

pat = re.compile(r"^(?P<base>.+)_(?P<num>[1-9]\d*)\.mp4$", re.IGNORECASE)

def list_basenames_with_parts(group_path):
    parts = defaultdict(list)
    for fn in os.listdir(group_path):
        if not fn.lower().endswith(".mp4"):
            continue
        m = pat.match(fn)
        if not m:
            continue
        base = m.group("base")
        num = int(m.group("num"))
        parts[base].append((num, os.path.join(group_path, fn)))
    for base in list(parts.keys()):
        parts[base].sort(key=lambda x: x[0])
    return parts

def get_layout(mode, count):
    if mode == "row":
        return (1, count)      # 1 x 5
    elif mode == "2x3":
        return (2, 3)          # 2 x 3
    else:
        raise ValueError("LAYOUT_MODE must be 'row' or '2x3'")

def letterbox(frame, target_w, target_h, bg_color):
    h, w = frame.shape[:2]
    scale = min(target_w / w, target_h / h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.full((target_h, target_w, 3), bg_color, dtype=np.uint8)
    y0 = (target_h - new_h) // 2
    x0 = (target_w - new_w) // 2
    canvas[y0:y0+new_h, x0:x0+new_w] = resized
    return canvas

def draw_label(img, text):
    """Draw a semi-transparent label box with text in top-left of the tile."""
    if not DRAW_LABELS or text is None:
        return img
    overlay = img.copy()

    # autoscale font relative to cell size
    base = min(img.shape[0], img.shape[1])
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = max(0.5, base / 240.0)     # tweak as desired
    thickness = max(1, int(round(base / 200.0)))

    # get text size
    (tw, th), baseline = cv2.getTextSize(text, font, font_scale, thickness)
    box_w = tw + 2 * LABEL_PADDING
    box_h = th + baseline + 2 * LABEL_PADDING

    x, y = LABEL_MARGIN, LABEL_MARGIN  # top-left corner of box
    # draw semi-transparent filled rectangle on overlay
    cv2.rectangle(overlay, (x, y), (x + box_w, y + box_h), (0, 0, 0), -1)
    # blend overlay with original
    cv2.addWeighted(overlay, LABEL_BG_ALPHA, img, 1 - LABEL_BG_ALPHA, 0, dst=img)

    # draw text in white
    tx = x + LABEL_PADDING
    ty = y + LABEL_PADDING + th
    cv2.putText(img, text, (tx, ty), font, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)
    return img

def stack_grid(frames, rows, cols, cell_w, cell_h, gutter, bg_color):
    total_w = cols * cell_w + (cols - 1) * gutter
    total_h = rows * cell_h + (rows - 1) * gutter
    canvas = np.full((total_h, total_w, 3), bg_color, dtype=np.uint8)
    idx = 0
    for r in range(rows):
        for c in range(cols):
            y = r * (cell_h + gutter)
            x = c * (cell_w + gutter)
            if idx < len(frames) and frames[idx] is not None:
                canvas[y:y+cell_h, x:x+cell_w] = frames[idx]
            idx += 1
    return canvas

def build_grid_video(sources, out_path, layout_mode):
    rows, cols = get_layout(layout_mode, EXPECTED_COUNT)
    slots = rows * cols

    # labels per slot: "1".."len(sources)" then None
    slot_labels = [str(i+1) if i < len(sources) else None for i in range(slots)]

    # open caps (pad to slot count with None)
    caps = []
    for i in range(slots):
        if i < len(sources):
            cap = cv2.VideoCapture(sources[i])
            if not cap.isOpened():
                print(f"⚠ Can't open: {sources[i]}")
                cap = None
        else:
            cap = None
        caps.append(cap)

    base_cap = next((c for c in caps if c is not None), None)
    if base_cap is None:
        return False, "No readable inputs"

    # choose fps from first valid cap (fallback 30)
    fps = base_cap.get(cv2.CAP_PROP_FPS) or 30.0
    for c in caps:
        if c is not None and (c.get(cv2.CAP_PROP_FPS) or 0) > 0:
            fps = c.get(cv2.CAP_PROP_FPS)
            break

    total_w = cols * TARGET_CELL_W + (cols - 1) * GUTTER
    total_h = rows * TARGET_CELL_H + (rows - 1) * GUTTER

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    writer = cv2.VideoWriter(out_path, FOURCC, fps, (total_w, total_h))
    if not writer.isOpened():
        for c in caps:
            if c is not None: c.release()
        return False, f"Failed to open writer: {out_path}"

    # keep last good frame per slot (letterboxed and labeled), for freeze-after-finish
    last_frames = [None] * len(caps)

    total_written = 0
    pbar = tqdm(desc=f"Grid -> {os.path.basename(out_path)}", unit="f")
    while True:
        any_new_frame = False
        cell_frames = []
        for idx, cap in enumerate(caps):
            label_text = slot_labels[idx]
            if cap is None:
                # no source in this slot -> use black (still show label if you want)
                filler = np.full((TARGET_CELL_H, TARGET_CELL_W, 3), BG_COLOR, dtype=np.uint8)
                cell_frames.append(draw_label(filler, label_text))
                continue

            ok, frame = cap.read()
            if ok:
                any_new_frame = True
                boxed = letterbox(frame, TARGET_CELL_W, TARGET_CELL_H, BG_COLOR)
                boxed = draw_label(boxed, label_text)
                last_frames[idx] = boxed
                cell_frames.append(boxed)
            else:
                # freeze on the last good frame; if none yet, use black with label
                if last_frames[idx] is not None:
                    cell_frames.append(last_frames[idx])
                else:
                    filler = np.full((TARGET_CELL_H, TARGET_CELL_W, 3), BG_COLOR, dtype=np.uint8)
                    cell_frames.append(draw_label(filler, label_text))

        # stop when NO input produced a new frame this iteration
        if not any_new_frame:
            break

        grid = stack_grid(cell_frames, rows, cols, TARGET_CELL_W, TARGET_CELL_H, GUTTER, BG_COLOR)
        writer.write(grid)
        total_written += 1
        pbar.update(1)

    pbar.close()
    for c in caps:
        if c is not None: c.release()
    writer.release()
    return True, f"Wrote {total_written} frames @ ~{fps:.2f} FPS, size {total_w}x{total_h}"

def process_group(group):
    group_path = os.path.join(VIDEOS_ROOT, group)
    out_dir = os.path.join(group_path, OUT_SUBFMT.format(group=group))
    os.makedirs(out_dir, exist_ok=True)

    parts_map = list_basenames_with_parts(group_path)
    if not parts_map:
        print(f"No numbered inputs found in {group_path}")
        return

    for base, numbered in parts_map.items():
        numbered.sort(key=lambda x: x[0])
        wanted = [path for num, path in numbered if 1 <= num <= EXPECTED_COUNT]
        if len(wanted) < EXPECTED_COUNT:
            print(f"⚠ {group}/{base}: only {len(wanted)} of {EXPECTED_COUNT} parts found (missing will be black)")
        out_path = os.path.join(out_dir, f"{base}.mp4")
        ok, msg = build_grid_video(wanted, out_path, LAYOUT_MODE)
        print(("✅" if ok else "❌") + f" {group}/{base}: {msg} -> {out_path}")

for group in GROUPS:
    if not os.path.isdir(os.path.join(VIDEOS_ROOT, group)):
        print(f"Skip (no folder): {os.path.join(VIDEOS_ROOT, group)}")
        continue
    process_group(group)


Grid -> hole.mp4: 350f [00:03, 94.13f/s] 


✅ step/hole: Wrote 350 frames @ ~16.00 FPS, size 1362x582 -> videos\step\combine_step\hole.mp4


Grid -> normal.mp4: 350f [00:03, 101.31f/s]


✅ step/normal: Wrote 350 frames @ ~16.00 FPS, size 1362x582 -> videos\step\combine_step\normal.mp4


Grid -> ramp.mp4: 350f [00:03, 97.50f/s] 


...

✅ randla/ramp_obstacle: Wrote 350 frames @ ~16.00 FPS, size 1362x582 -> videos\randla\combine_randla\ramp_obstacle.mp4


Grid -> two_height_ramp.mp4: 350f [00:02, 123.56f/s]


✅ randla/two_height_ramp: Wrote 350 frames @ ~16.00 FPS, size 1362x582 -> videos\randla\combine_randla\two_height_ramp.mp4


Grid -> uneven.mp4: 350f [00:03, 99.19f/s] 

✅ randla/uneven: Wrote 350 frames @ ~16.00 FPS, size 1362x582 -> videos\randla\combine_randla\uneven.mp4
